# Gallery Example — Manhattan Plot

This notebook builds a GenomeSpy-native Manhattan plot. The synthetic GWAS table follows the same data shape and deterministic random seed as the `build_gwas()` helper in `tmp/dataviz-genomicsdata`, while the visual target is inspired by `ggwas`.

What to eyeball after rendering:

- chromosome labels and boundary ticks should read cleanly across the whole genome
- alternating chromosome point colors should help separate blocks without dominating the view, with a legend for odd/even chromosome groups
- the genome-wide and suggestive threshold rules should be obvious but subtle
- the strongest hits should stand out as larger outlined peak markers without changing the chromosome color meaning

In [1]:
import math

import numpy as np
import pandas as pd

import genome_spy as gs
from genome_spy.schema import GenomeAxis, Scale

rng = np.random.default_rng(7)
chrom_sizes = {chrom: int(900 - chrom * 25) for chrom in range(1, 23)}

frames = []
for chrom, size in chrom_sizes.items():
    positions = np.sort(rng.integers(1, 250_000_000, size))
    pvals = rng.uniform(0, 1, size)
    frames.append(
        pd.DataFrame(
            {
                "chrom": [f"chr{chrom}"] * size,
                "chrom_index": chrom,
                "pos": positions,
                "pval": pvals,
            }
        )
    )

source = pd.concat(frames, ignore_index=True)

hit_idx = rng.choice(len(source), 12, replace=False)
source.loc[hit_idx, "pval"] = 10 ** (-rng.uniform(8, 30, 12))
source["snp"] = [f"rs{rng.integers(100000, 9000000)}" for _ in range(len(source))]
source["neglog"] = -np.log10(source["pval"].clip(lower=1e-300))
source["chrom_group"] = np.where(source["chrom_index"] % 2 == 0, "even", "odd")

top_hits = source.nsmallest(5, "pval").copy()

genome_wide_threshold = -math.log10(5e-8)
suggestive_threshold = -math.log10(1e-5)

source.head()

,chrom,chrom_index,pos,pval,snp,neglog,chrom_group
0,chr1,1,562879,0.197842,rs1217797,0.703681,odd
1,chr1,1,802716,0.454291,rs3152883,0.342666,odd
2,chr1,1,933561,0.750287,rs1966161,0.124773,odd
3,chr1,1,1294717,0.707315,rs2836508,0.150387,odd
4,chr1,1,1316327,0.553459,rs6514392,0.256915,odd


In [4]:
axis = GenomeAxis(
    title="Genomic position",
    chromGrid=True,
    chromGridOpacity=0.14,
    chromGridFillEven="#f4f6fb",
    chromGridFillOdd="#ffffff",
    chromLabels=True,
    chromLabelFontSize=11,
    chromTicks=True,
    chromTickSize=10,
    labelFontSize=10,
    grid=False,
)

points = (
    gs.Chart(source)
    .mark_point(size=20, filled=True, opacity=0.82)
    .encode(
        x=gs.Locus("chrom", "pos", scale={"assembly": "hg38"}, axis=axis),
        y=gs.Y("neglog:Q").scale(reverse=False, zero=True).title("−log10 p"),
        color=gs.Color("chrom_group:N")
        .scale(Scale(range=["#5b8fd6", "#8f98a3"]))
        .legend(title="Chromosome group"),
    )
)

genome_wide_rule = (
    gs.Chart([{"threshold": genome_wide_threshold}])
    .mark_rule(strokeDash=[6, 4], size=1.4, color="#c53b2c")
    .encode(y=gs.Y("threshold:Q").scale(reverse=False))
)

suggestive_rule = (
    gs.Chart([{"threshold": suggestive_threshold}])
    .mark_rule(strokeDash=[2, 4], size=1.2, color="#d48b31")
    .encode(y=gs.Y("threshold:Q").scale(reverse=False))
)

highlight_points = (
    gs.Chart(top_hits)
    .mark_point(size=48, filled=True, stroke="black", strokeWidth=0.5)
    .encode(
        x=gs.Locus("chrom", "pos", scale={"assembly": "hg38"}),
        y=gs.Y("neglog:Q").scale(reverse=False),
        color=gs.Color("chrom_group:N")
        .scale(Scale(range=["#5b8fd6", "#8f98a3"]))
        .legend(None),
    )
)

chart = (genome_wide_rule + suggestive_rule + points + highlight_points).properties(
    width=500,
    height=280,
    title="Synthetic height GWAS across the genome",
    description="A GenomeSpy-native Manhattan plot with locus-aware chromosome axis and significance rules.",
)

chart

In [3]:
chart.spec

{
  "description": "A GenomeSpy-native Manhattan plot with locus-aware chromosome axis and significance rules.",
  "height": 380,
  "layer": [
    {
      "data": {
        "values": [
          {
            "threshold": 7.301029995663981
          }
        ]
      },
      "encoding": {
        "y": {
          "field": "threshold",
          "type": "quantitative",
          "scale": {
            "reverse": false
          }
        }
      },
      "mark": {
        "type": "rule",
        "strokeDash": [
          6,
          4
        ],
        "size": 1.4,
        "color": "#c53b2c"
      }
    },
    {
      "data": {
        "values": [
          {
            "threshold": 5.0
          }
        ]
      },
      "encoding": {
        "y": {
          "field": "threshold",
          "type": "quantitative",
          "scale": {
            "reverse": false
          }
        }
      },
      "mark": {
        "type": "rule",
        "strokeDash": [
          2,
          4